In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw"))

In [0]:
from pyspark.sql import functions as F
base = "/Volumes/workspace/default/raw"

# ---------- BRONZE: raw CSVs as-is (columns show as _c0, _c1, ...) ----------
def bronze(file, table):
    (spark.read.option("header", False).option("sep", "\t").csv(f"{base}/{file}")
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"workspace.default.{table}"))

bronze("PurchaseOrderDetail.csv", "stg_purchase_order_detail")
bronze("PurchaseOrderHeader.csv", "stg_purchase_order_header")
bronze("Vendor.csv", "dim_vendor")
display(spark.table("workspace.default.stg_purchase_order_detail").limit(3))

# ---------- SILVER: real column names + types + cleaning ----------
def silver(table, cols, ints=(), decs=(), tss=()):
    df = spark.table(f"workspace.default.{table}").toDF(*cols)
    for c in ints: df = df.withColumn(c, F.col(c).cast("int"))
    for c in decs: df = df.withColumn(c, F.col(c).cast("double"))
    for c in tss:  df = df.withColumn(c, F.col(c).cast("timestamp"))
    return df

detail = silver("stg_purchase_order_detail",
    ["PurchaseOrderID","PurchaseOrderDetailID","DueDate","OrderQty","ProductID","UnitPrice",
     "LineTotal","ReceivedQty","RejectedQty","StockedQty","ModifiedDate"],
    ints=["PurchaseOrderID","PurchaseOrderDetailID","OrderQty","ProductID"],
    decs=["UnitPrice","LineTotal","ReceivedQty","RejectedQty","StockedQty"],
    tss=["DueDate","ModifiedDate"])
detail = detail.filter(F.col("ProductID").isNotNull() & (F.col("LineTotal") >= 0))

header = silver("stg_purchase_order_header",
    ["PurchaseOrderID","RevisionNumber","Status","EmployeeID","VendorID","ShipMethodID",
     "OrderDate","ShipDate","SubTotal","TaxAmt","Freight","TotalDue","ModifiedDate"],
    ints=["PurchaseOrderID","RevisionNumber","Status","EmployeeID","VendorID","ShipMethodID"],
    decs=["SubTotal","TaxAmt","Freight","TotalDue"],
    tss=["OrderDate","ShipDate","ModifiedDate"])

vendor = silver("dim_vendor",
    ["VendorID","AccountNumber","Name","CreditRating","PreferredVendorStatus","ActiveFlag",
     "PurchasingWebServiceURL","ModifiedDate"],
    ints=["VendorID","CreditRating","PreferredVendorStatus","ActiveFlag"],
    tss=["ModifiedDate"])

for df, t in [(detail,"stg_purchase_order_detail"),(header,"stg_purchase_order_header"),(vendor,"dim_vendor")]:
    df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable(f"workspace.default.{t}")
    print(t, spark.table(f"workspace.default.{t}").count())

In [0]:
detail = spark.table("workspace.default.stg_purchase_order_detail")
header = spark.table("workspace.default.stg_purchase_order_header")

# Product names from the AdventureWorks Product.csv (saved as "Product (2).csv")
prod = (spark.read.option("header", False).option("sep", "\t")
        .csv(f"{base}/Product (2).csv")
        .select(F.col("_c0").cast("int").alias("ProductID"),
                F.trim(F.col("_c1")).alias("ProductName")))

fact_purchases = (detail
    .join(header.select("PurchaseOrderID", "VendorID", "OrderDate"), "PurchaseOrderID", "inner")
    .join(prod, "ProductID", "left")
    .select("PurchaseOrderID", "PurchaseOrderDetailID", "VendorID", "OrderDate",
            "ProductID", "ProductName", "OrderQty", "UnitPrice", "LineTotal",
            "ReceivedQty", "RejectedQty", "StockedQty"))

fact_purchases.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.fact_purchases")

print("fact_purchases rows:", spark.table("workspace.default.fact_purchases").count())
print("detail rows:        ", detail.count())
print("rows without a product name:",
      spark.table("workspace.default.fact_purchases").filter("ProductName IS NULL").count())
display(spark.table("workspace.default.fact_purchases").limit(5))

In [0]:
%sql
SHOW TABLES IN workspace.default

In [0]:
%sql
WITH s AS (
  SELECT TRIM(p.Product) AS product_name,
         AVG(f.Unit_Price) AS avg_sales_price,
         SUM(f.Quantity)   AS units_sold
  FROM fact_sales f JOIN dim_product p ON f.ProductKey = p.ProductKey
  GROUP BY TRIM(p.Product)
),
pu AS (
  SELECT ProductName AS product_name,
         AVG(UnitPrice) AS avg_purchase_price,
         SUM(OrderQty)  AS units_purchased
  FROM fact_purchases
  GROUP BY ProductName
)
SELECT s.product_name,
       ROUND(s.avg_sales_price, 2)    AS avg_sales_price,
       ROUND(pu.avg_purchase_price, 2) AS avg_purchase_price,
       ROUND(s.avg_sales_price - pu.avg_purchase_price, 2) AS margin
FROM s JOIN pu ON s.product_name = pu.product_name
ORDER BY margin ASC
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW monthly_flow AS
WITH s AS (
  SELECT TRIM(p.Product) AS product_name, date_trunc('month', f.OrderDate) AS month,
         SUM(f.Quantity) AS units_sold
  FROM fact_sales f JOIN dim_product p ON f.ProductKey = p.ProductKey
  GROUP BY TRIM(p.Product), date_trunc('month', f.OrderDate)
),
pu AS (
  SELECT ProductName AS product_name, date_trunc('month', OrderDate) AS month,
         SUM(OrderQty) AS units_purchased
  FROM fact_purchases
  GROUP BY ProductName, date_trunc('month', OrderDate)
),
m AS (
  SELECT COALESCE(s.product_name, pu.product_name) AS product_name,
         COALESCE(s.month, pu.month) AS month,
         COALESCE(s.units_sold, 0) AS units_sold,
         COALESCE(pu.units_purchased, 0) AS units_purchased
  FROM s FULL OUTER JOIN pu ON s.product_name = pu.product_name AND s.month = pu.month
)
SELECT * FROM m
WHERE product_name IN (SELECT product_name FROM s)
  AND product_name IN (SELECT product_name FROM pu)

In [0]:
%sql
SELECT product_name, month, units_sold, units_purchased,
       units_sold - units_purchased AS net
FROM monthly_flow
ORDER BY product_name, month

In [0]:
%sql
SELECT product_name,
       SUM(units_sold)      AS total_sold,
       SUM(units_purchased) AS total_purchased,
       SUM(units_sold) - SUM(units_purchased) AS net,
       CASE WHEN SUM(units_sold) > SUM(units_purchased)
            THEN 'sells faster than restocked' ELSE 'restocked at least as fast as sold' END AS status
FROM monthly_flow
GROUP BY product_name
ORDER BY net DESC